# 📊 Monitoring and Logging for Production## Learning ObjectivesIn this notebook, you will learn:1. **Structured JSON Logging** - build a `JSONFormatter` and a logger factory that emit machine-parseable logs for aggregation tools like Datadog or the ELK stack.2. **Metrics Collection** - track request counts, error rates, latency, token usage, and cache hit rate with a lightweight `MetricsCollector`.3. **LLM Instrumentation** - wrap an LLM call in an `InstrumentedLLM` class that combines logging, metrics, and LangSmith tracing in one place.4. **Observability in Production** - see how logs, metrics, and traces complement each other to answer "what happened", "how often", and "why" for a live LLM application.## Prerequisites- An `OPENAI_API_KEY` set in a `.env` file at the project root (loaded via `python-dotenv`).- Optional LangSmith credentials (`LANGCHAIN_TRACING_V2`, `LANGCHAIN_API_KEY`) if you want the `@traceable` decorator to emit traces to LangSmith.- Basic familiarity with Python's `logging` module and LangChain chat models.

---## 🔧 Part 0: Environment SetupBefore instrumenting anything, we load environment variables and import the libraries this notebook relies on: Python's standard `logging`/`json` modules for structured logs, LangChain's `ChatOpenAI` for the model under observation, and `langsmith`'s `traceable` decorator for distributed tracing. Centralizing these imports up front keeps every section below focused on monitoring logic rather than setup.

In [ ]:
# ============================================================================# ENVIRONMENT SETUP: Imports and Configuration# ============================================================================import jsonimport loggingimport timefrom datetime import datetime, timezonefrom functools import wrapsfrom typing import Any, Callablefrom dotenv import load_dotenvfrom langsmith import traceablefrom langchain_core.callbacks import BaseCallbackHandlerfrom langchain_core.messages import HumanMessagefrom langchain_openai import ChatOpenAIload_dotenv()print("✅ Environment ready: logging, metrics, and tracing dependencies loaded.")

---## 📝 Part 1: Structured LoggingPlain-text logs are easy for humans but hard for machines to search and aggregate at scale. This section defines a `JSONFormatter` that renders every log record as a single JSON object, plus a `setup_logging` helper that wires it into a named logger — the foundation any log aggregation pipeline (Datadog, ELK, CloudWatch Logs Insights) can ingest directly.

### 🧩 `JSONFormatter` — JSON Log FormatterSubclasses `logging.Formatter` to emit each record as a JSON string containing a UTC timestamp, level, message, module, and function name. Any extra context passed via `extra={"extra_data": {...}}` on a log call is merged into the same object, so downstream consumers get structured fields instead of parsing free-text messages.

In [ ]:
# ============================================================================# JSONFORMATTER: JSON Log Formatter for Structured Logging# ============================================================================class JSONFormatter(logging.Formatter):    """Format logs as JSON for log aggregation."""    def format(self, record):        log_obj = {            "timestamp": datetime.now(timezone.utc).isoformat(),            "level": record.levelname,            "message": record.getMessage(),            "module": record.module,            "function": record.funcName,        }        if hasattr(record, "extra_data"):            log_obj.update(record.extra_data)        return json.dumps(log_obj)

### ⚙️ `setup_logging` — Wire Up the LoggerCreates (or reuses) a named `langgraph_app` logger, attaches a `StreamHandler`, and sets its formatter to the `JSONFormatter` above. Calling this once per process gives every subsequent `logger.info(...)` / `logger.error(...)` call structured JSON output.

In [ ]:
# ============================================================================# SETUP_LOGGING: Configure the Structured Logger# ============================================================================def setup_logging():    """Setup structured JSON logging."""    logger = logging.getLogger("langgraph_app")    logger.setLevel(logging.INFO)    handler = logging.StreamHandler()    handler.setFormatter(JSONFormatter())    logger.addHandler(handler)    return logger

---## 📈 Part 2: Metrics CollectionLogs answer "what happened for this one request"; metrics answer "how is the system doing overall." `MetricsCollector` accumulates simple counters — requests, errors, latency, tokens, cache hits/misses — in memory and exposes a `get_summary()` method that derives the rates (error rate, average latency, cache hit rate) a dashboard or alert would care about.### Key Concepts:- **Counter vs. derived metric**: raw counts (`requests_total`, `latency_sum`) are cheap to update on every call; rates (`error_rate`, `cache_hit_rate`) are computed on demand in `get_summary()`.- **Guarded division**: every ratio in `get_summary()` checks its denominator is non-zero before dividing, so the summary is safe to call before any traffic has been recorded.

### 📦 `MetricsCollector` — Aggregate Runtime MetricsTracks eight running counters in a single `dict` and exposes `record_request()` to update them after every LLM call, plus `get_summary()` to produce a human-readable snapshot (error rate, average latency in ms, total tokens, cache hit rate).

In [ ]:
# ============================================================================# METRICS_COLLECTOR: Aggregate Request, Latency, and Token Metrics# ============================================================================class MetricsCollector:    """Collect and aggregate metrics."""    def __init__(self):        self.metrics = {            "requests_total": 0,            "errors_total": 0,            "latency_sum": 0,            "latency_count": 0,            "tokens_input": 0,            "tokens_output": 0,            "cache_hits": 0,            "cache_misses": 0,        }    def record_request(        self,        latency_ms: float,        input_tokens: int,        output_tokens: int,        error: bool = False,        cache_hit: bool = False,    ):        self.metrics["requests_total"] += 1        self.metrics["latency_sum"] += latency_ms        self.metrics["latency_count"] += 1        self.metrics["tokens_input"] += input_tokens        self.metrics["tokens_output"] += output_tokens        if error:            self.metrics["errors_total"] += 1        if cache_hit:            self.metrics["cache_hits"] += 1        else:            self.metrics["cache_misses"] += 1    def get_summary(self) -> dict:        avg_latency = (            self.metrics["latency_sum"] / self.metrics["latency_count"]            if self.metrics["latency_count"] > 0            else 0        )        error_rate = (            self.metrics["errors_total"] / self.metrics["requests_total"]            if self.metrics["requests_total"] > 0            else 0        )        cache_hit_rate = (            self.metrics["cache_hits"]            / (self.metrics["cache_hits"] + self.metrics["cache_misses"])            if (self.metrics["cache_hits"] + self.metrics["cache_misses"]) > 0            else 0        )        return {            "total_requests": self.metrics["requests_total"],            "total_errors": self.metrics["errors_total"],            "error_rate": f"{error_rate:.2%}",            "avg_latency_ms": round(avg_latency, 2),            "total_input_tokens": self.metrics["tokens_input"],            "total_output_tokens": self.metrics["tokens_output"],            "cache_hit_rate": f"{cache_hit_rate:.2%}",        }

---## 🤖 Part 3: Instrumented LLM WrapperWith a logger and a metrics collector in hand, `InstrumentedLLM` ties them together with LangSmith's `@traceable` decorator so a single `invoke()` call produces a structured log line, updates the in-memory metrics, and emits a trace — without any of that instrumentation code leaking into the call sites that use the LLM.

### 🔌 `InstrumentedLLM` — Logging + Metrics + Tracing in One WrapperWraps a `ChatOpenAI` instance behind a `@traceable`-decorated `invoke()` method. On success it estimates input/output token counts, records the request in `MetricsCollector`, and logs a JSON "LLM request completed" line; on failure it records the error in metrics, logs the exception, and re-raises so callers still see failures.> **Note**: Token counts here are a rough word-count-based estimate (`len(text.split()) * 4 // 3`), not an exact tokenizer count — good enough for monitoring trends, not for billing-accurate reporting.

In [ ]:
# ============================================================================# INSTRUMENTED_LLM: Logging + Metrics + LangSmith Tracing Wrapper# ============================================================================class InstrumentedLLM:    """LLM with full instrumentation."""    def __init__(self):        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)        self.metrics = MetricsCollector()        self.logger = setup_logging()    @traceable(name="instrumented_invoke")    def invoke(self, query: str) -> str:        start_time = time.time()        error = False        try:            response = self.llm.invoke(query)            result = response.content            # Estimate tokens            input_tokens = len(query.split()) * 4 // 3            output_tokens = len(result.split()) * 4 // 3            self.metrics.record_request(                latency_ms=(time.time() - start_time) * 1000,                input_tokens=input_tokens,                output_tokens=output_tokens,                error=False,                cache_hit=False,            )            self.logger.info(                "LLM request completed",                extra={                    "extra_data": {                        "latency_ms": (time.time() - start_time) * 1000,                        "input_tokens": input_tokens,                        "output_tokens": output_tokens,                    }                },            )            return result        except Exception as e:            error = True            self.metrics.record_request(                latency_ms=(time.time() - start_time) * 1000,                input_tokens=0,                output_tokens=0,                error=True,                cache_hit=False,            )            self.logger.error(                f"LLM request failed: {e}", extra={"extra_data": {"error": str(e)}}            )            raise

---## 🚀 Part 4: DemoFinally, `demo_monitoring()` exercises the full stack end to end: it creates an `InstrumentedLLM`, runs a handful of sample queries through it, and prints the aggregated metrics summary so you can see logging, metrics, and tracing working together on real output.

### ▶️ `demo_monitoring` — End-to-End DemonstrationSends three sample queries through the instrumented LLM, printing a truncated preview of each query/response pair, then prints the final `MetricsCollector` summary (total requests, error rate, average latency, token totals, cache hit rate).

In [ ]:
# ============================================================================# DEMO_MONITORING: End-to-End Monitoring Demonstration# ============================================================================def demo_monitoring():    """Demonstrate monitoring."""    llm = InstrumentedLLM()    print("Monitoring Demo:\n")    queries = [        "What is Python?",        "Explain machine learning.",        "What is 2 + 2?",    ]    for query in queries:        result = llm.invoke(query)        print(f"Query: {query[:30]}... -> {result[:30]}...")    print("\nMetrics Summary:")    summary = llm.metrics.get_summary()    for key, value in summary.items():        print(f"  {key}: {value}")

---## ▶️ Part 5: Execute the DemoThe original script's `__main__` guard is kept verbatim below. Jupyter sets `__name__` to `"__main__"` in a notebook, so this cell runs as-is when executed — running `demo_monitoring()` by default. Uncomment the commented lines instead if you'd rather just exercise `setup_logging()` directly.

In [ ]:
# ============================================================================# RUN: Execute the Monitoring Demo# ============================================================================if __name__ == "__main__":    # logger = setup_logging()    # logger.info("Logging setup complete", extra={"extra_data": {"app": "langgraph"}})    demo_monitoring()

---## 📝 SummaryThis notebook built a minimal but complete observability stack for an LLM application, combining structured logs, aggregate metrics, and distributed tracing.### 1. Structured Logging- **`JSONFormatter`**: renders every log record as a single JSON object with timestamp, level, message, module, and function.- **`setup_logging`**: attaches that formatter to a named logger so every call site gets consistent, aggregator-friendly output.### 2. Metrics Collection- **`MetricsCollector`**: accumulates request counts, errors, latency, token usage, and cache hits/misses.- **`get_summary()`**: derives error rate, average latency, and cache hit rate on demand, guarding against divide-by-zero before any traffic exists.### 3. Instrumented LLM Calls- **`InstrumentedLLM`**: wraps `ChatOpenAI.invoke()` with a `@traceable` decorator, logging, and metrics recording in a single call path.- **`demo_monitoring()`**: exercises the full stack against three sample queries and prints the resulting metrics summary.### Next Steps- Swap the `StreamHandler` in `setup_logging` for a handler that ships logs to your aggregator of choice (Datadog, CloudWatch, ELK).- Replace the word-count token estimate in `InstrumentedLLM.invoke` with your provider's actual token usage (e.g., `response.usage_metadata`) for billing-accurate metrics.- Wire `MetricsCollector.get_summary()` into a real metrics backend (Prometheus, StatsD) and set alert thresholds on `error_rate` and `avg_latency_ms`.- Continue to the next notebook in `05_Production_and_Operations/` for alerting and incident-response patterns.